# Setup
## Dependencies

In [ ]:
%config InlineBackend.figure_format = 'retina'
from astropy.io import fits
from astropy import units as u
from astropy.coordinates import SkyCoord
import numpy as np
import matplotlib.pyplot as plt
from astropy.wcs import WCS
from photutils.centroids import centroid_quadratic
from photutils.aperture import CircularAnnulus, CircularAperture, ApertureStats, aperture_photometry


## etc.

In [ ]:
supernova_name = 'SN2026aaiv'

#mosaic_image = fits.open('../non_flat_mosaics/SN2026aaiv/sn2026aaiv-26_10_02-pt5m_V-mosaic.fits')
mosaic_image = fits.open('../output mosaics/non-flats/at2026aesj-26_10_13-pt5m-B-mosaic.fits')
mosaic_image_data = mosaic_image[0].data
mosaic_image_header = mosaic_image[0].header
wcs = WCS(mosaic_image_header)



exp_time = mosaic_image_header['EXPTIME']
#print(exp_time)
#print(mosaic_image_header)

aaiv_RA_Dec =  SkyCoord('22h37m05s', '+34d24m35.37s')
aaiv_RA_Dec = SkyCoord("07h48m40s", "+54d36m43s")
#aaiv_RA_Dec = SkyCoord('22h37m08.472s', '+34d26m12.05s')
aaiv_x, aaiv_y = wcs.world_to_pixel(aaiv_RA_Dec)



clean_data = np.nan_to_num(mosaic_image_data, nan=0.0)
mosaic_image_data = clean_data
#mosaic_image.info()

#R.A. = 22h37m05s.618, Decl. = +34°24'35".37

## Load Image

In [ ]:
img_mean, img_sigma = np.mean(mosaic_image_data), np.std(mosaic_image_data)

plt.figure()
plt.imshow(
    mosaic_image_data,
    vmin = img_mean - 1 * img_sigma,
    vmax =  img_mean + 5 * img_sigma,
    cmap = 'grey',
    origin='lower',
    interpolation='nearest',
)

plt.xlabel("x Position (Pixels)")
plt.ylabel("y Position (Pixels)")
plt.title(f"{supernova_name} Quick Look")
plt.savefig(f'../graphs/{supernova_name}-Quick_Look.png', format='png', dpi=300, bbox_inches='tight')
plt.show()

## Supernova Centroid

In [ ]:
supernova_xy_center = centroid_quadratic(
    mosaic_image_data, 
    xpeak = aaiv_x, 
    ypeak = aaiv_y,
    search_boxsize=21, 
    fit_boxsize=9,
)



print(supernova_xy_center)



plt.figure()
plt.imshow(
    mosaic_image_data,
    vmin = img_mean - 1 * img_sigma,
    vmax =  img_mean + 5 * img_sigma,
    cmap = 'grey',
    origin='lower',
    interpolation='nearest',
)

plt.scatter(supernova_xy_center[0], supernova_xy_center[1], 
            s=30, color='red', marker='+', linewidth = 0.5,
           label='Centroid',
)






plt.xlabel("x Position (Pixels)")
plt.ylabel("y Position (Pixels)")
plt.title(f"{supernova_name} Centroids")
plt.savefig(f'../graphs/{supernova_name}-Centroids.png', format='png', dpi=300, bbox_inches='tight')
plt.show()

# Aperture Photometry
## Zoomed Section/Aperture and Annulus test

In [ ]:
aperture_radius = 12
annuli_inner = 18
annuli_outer = 24

zoom_height = 200
zoom_width = zoom_height * 1.5
section_ymin, section_ymax = (supernova_xy_center[1] - zoom_height/2), (supernova_xy_center[1] + zoom_height/2)
section_xmin, section_xmax = (supernova_xy_center[0] - zoom_width/2), (supernova_xy_center[0] + zoom_width/2)
section = mosaic_image_data[int(section_ymin):int(section_ymax), int(section_xmin):int(section_xmax)]

apertures = CircularAperture((supernova_xy_center), r=aperture_radius)
annuli = CircularAnnulus((supernova_xy_center), r_in=annuli_inner, r_out=annuli_outer)

plt.imshow(
    mosaic_image_data,
    vmin = img_mean - 1 * img_sigma,
    vmax =  img_mean + 5 * img_sigma,
    cmap = 'grey',
    origin='lower',
    interpolation='nearest',
)
apertures.plot(color = 'blue', lw = .7, alpha = 1);
annuli.plot(color = 'red', lw = .7, alpha = 1);
plt.xlabel('X position (pixels)')
plt.ylabel('Y position (pixels)')
cbar = plt.colorbar()
cbar.set_label('Counts (ADU)')
#plt.title(f'{Star_Name} and {Comp_Star_Name} (Green Channel)')


plt.xlim(section_xmin,section_xmax)
plt.ylim(section_ymax,section_ymin)
#plt.savefig(f'../graphs/{Star_Name}_{Comp_Star_Name}-Zoomed_Image.png', dpi=300, bbox_inches='tight')

plt.scatter(supernova_xy_center[0], supernova_xy_center[1], 
            s=30, color='red', marker='+', linewidth = 0.5,
           label='Centroid',
)
plt.show()


In [ ]:
aperture = CircularAperture(supernova_xy_center, r=aperture_radius)
annuli = CircularAnnulus(supernova_xy_center, r_in=annuli_inner, r_out=annuli_outer)

phot_table = aperture_photometry(mosaic_image_data, [apertures, annuli]) 

C_aper = phot_table['aperture_sum_0'] 
C_annul = phot_table['aperture_sum_1'] 
N_aper = apertures.area
N_annul = annuli.area 

C_sky = N_aper * (C_annul/N_annul)
C_obj = C_aper - C_sky


#print(C_aper)
#print(C_annul) 
#print(N_aper)
#print(N_annul)
#print(C_sky)
#print(float(C_obj))

#print(float(C_obj)/N_aper)
#print(float(C_obj)[1])

zeropoint = 24.584
mag_obj = -2.5 * (np.log10(C_obj)) + zeropoint
#mag_obj = float(mag_obj) * u.mag

print(f"Supernova Magnitude: {(mag_obj)[0]}")

#lsd;akfj;laskfjsd

def get_photometry(image_data, apertures, annuli, zeropoint):
    phot_table = aperture_photometry(image_data, [apertures, annuli]) 
    C_aper = phot_table['aperture_sum_0'] 
    C_annul = phot_table['aperture_sum_1'] 
    N_aper = apertures.area
    N_annul = annuli.area 

    C_sky = N_aper * (C_annul/N_annul)
    C_obj = C_aper - C_sky

    mag_obj = -2.5 * (np.log10(C_obj)) + zeropoint

    return(mag_obj)
    

## Aperture and Annulus Size Testing
### List of Sizes

In [ ]:

#%matplotlib widget
#%pip install --upgrade ipympl ipywidgets
aperture_sizes = list(range(8,21,1))
annuli_inn_sizes = list(range(8,500,1))
annuli_width = 20

mag_obj_list = []
aperture_size_list = []
annuli_inn_size_list = []

results = []

for aperture_rad in aperture_sizes:
    for annulus_inn in annuli_inn_sizes:
        if annulus_inn < aperture_rad:
            continue


        annulus_out = annulus_inn + annuli_width

        aperture = CircularAperture(supernova_xy_center, r=aperture_rad)
        annulus = CircularAnnulus(supernova_xy_center, r_in=annulus_inn, r_out=annulus_out)
            
            
        mag = get_photometry(mosaic_image_data, aperture, annulus, zeropoint)

        results.append((aperture_rad, annulus_inn, float(mag[0])))

In [ ]:
aperture_values, annulus_values, magnitudes = zip(*results)

ax = plt.figure().add_subplot(projection="3d")
ax.scatter(aperture_values, annulus_values, magnitudes)

ax.set_xlabel("Aperture radius (pixels)")
ax.set_ylabel("Inner annulus radius (pixels)")
ax.set_zlabel("Magnitude")

plt.show()

In [ ]:
import plotly.graph_objects as go

fig = go.Figure(data=[
   go.Scatter3d(
       x=[row[0] for row in results],
       y=[row[1] for row in results],
       z=[row[2] for row in results],
       mode="markers",
       marker=dict(size=4, color=[row[2] for row in results], colorscale="Viridis"),
   )
])

fig.update_layout(
   scene=dict(
       xaxis_title="Aperture radius (pixels)",
       yaxis_title="Inner annulus radius (pixels)",
       zaxis_title="Magnitude",
   ),
   #title="magnitude vs aperture and annulus size",
)

fig.write_html("aperture_test.html", auto_open=True
              )